# 00 — Project Setup and Environment Check

This notebook verifies that the clean Dutch PEP benchmark project is correctly configured before the data collection, cleaning, matching, and validation notebooks are run.

The notebook checks:

1. the project folder structure;
2. the active Python environment;
3. required Python packages;
4. required input files;
5. the presence of the OpenSanctions file when matching is later performed.

This notebook does not modify benchmark data.

In [ ]:
from pathlib import Path
import sys
import platform
import importlib.util
import pandas as pd

# -------------------------------------------------------------------
# Project folder setup
# -------------------------------------------------------------------
# The notebook may be run either from:
# - Code_clean/
# - Code_clean/notebooks/
#
# This block identifies the actual project root folder in both cases.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

INPUT_DIR = PROJECT_DIR / "data" / "input"
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
RAW_HTML_DIR = PROJECT_DIR / "data" / "raw_html"
RAW_TEXT_DIR = PROJECT_DIR / "data" / "raw_text"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"

# Create folders if they do not already exist.
for folder in [
    INPUT_DIR,
    EXTERNAL_DIR,
    OUTPUT_DIR,
    RAW_HTML_DIR,
    RAW_TEXT_DIR,
    NOTEBOOKS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Current working folder:", CURRENT_DIR)
print("Project folder:", PROJECT_DIR)

print("\nProject folders:")
print("Input:", INPUT_DIR)
print("External:", EXTERNAL_DIR)
print("Output:", OUTPUT_DIR)
print("Raw HTML:", RAW_HTML_DIR)
print("Raw text:", RAW_TEXT_DIR)
print("Notebooks:", NOTEBOOKS_DIR)

## 1. Check Python environment

This section confirms which Python interpreter and virtual environment are currently being used.

The expected interpreter should be located inside the local `.venv` folder.

In [ ]:
# -------------------------------------------------------------------
# Python environment check
# -------------------------------------------------------------------

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

print("\nOperating system:")
print(platform.platform())

using_local_venv = ".venv" in sys.executable.lower()

print("\nUsing local .venv:", using_local_venv)

if not using_local_venv:
    print(
        "\nWarning: the active Python interpreter does not appear to use "
        "the Code_clean/.venv environment."
    )
    print(
        "In VS Code, select the kernel or interpreter named "
        "'Thesis PEP Benchmark' or the Python executable inside .venv."
    )

## 2. Check required Python packages

This section checks whether the packages required by the notebooks are installed.

The project uses:

- `pandas` for data handling;
- `openpyxl` for Excel input files;
- `requests` and `beautifulsoup4` for source collection;
- `selenium` as a fallback for dynamic pages;
- `ipykernel` and `notebook` for running Jupyter notebooks.

In [ ]:
# -------------------------------------------------------------------
# Required package check
# -------------------------------------------------------------------

required_packages = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "requests": "requests",
    "beautifulsoup4": "bs4",
    "lxml": "lxml",
    "selenium": "selenium",
    "ipykernel": "ipykernel",
    "notebook": "notebook"
}

package_check_rows = []

for package_name, import_name in required_packages.items():
    installed = importlib.util.find_spec(import_name) is not None

    package_check_rows.append({
        "package": package_name,
        "import_name": import_name,
        "installed": installed
    })

package_check_df = pd.DataFrame(package_check_rows)

display(package_check_df)

missing_packages = package_check_df[
    ~package_check_df["installed"]
]["package"].tolist()

if missing_packages:
    print("Missing packages:")
    print(", ".join(missing_packages))

    print("\nInstall them in PowerShell with:")
    print(f"pip install {' '.join(missing_packages)}")
else:
    print("All required packages are installed.")

## 3. Check required project files

This section checks whether the manually prepared input files are available.

The OpenSanctions file is not required yet. It is needed only when running the matching and reverse-validation notebooks.

In [ ]:
# -------------------------------------------------------------------
# Required project file check
# -------------------------------------------------------------------

CURATED_INTERMEDIATES_DIR = INPUT_DIR / "curated_intermediates"

required_input_files = {
    "PEP source mapping": INPUT_DIR / "PEP_source_mapping.csv",
    "EU-aligned taxonomy": INPUT_DIR / "taxonomy_v2_eu_aligned.xlsx",
    "Manual benchmark additions": INPUT_DIR / "manual_main_v2_additions.xlsx",
    "Category C party sources": INPUT_DIR / "category_c_party_sources.csv",
    "Manual Category C records": INPUT_DIR / "category_c_party_board_manual_v3.csv",
    "Curated benchmark v1": (
        CURATED_INTERMEDIATES_DIR / "benchmark_v1_curated.csv"
    ),
}

optional_external_files = {
    "OpenSanctions target file": (
        EXTERNAL_DIR / "opensanctions_targets.simple.csv"
    )
}

# Check required input files.
required_file_check_rows = []

for file_name, file_path in required_input_files.items():
    required_file_check_rows.append({
        "file": file_name,
        "path": str(file_path),
        "required": True,
        "exists": file_path.exists()
    })

required_file_check_df = pd.DataFrame(required_file_check_rows)

display(required_file_check_df)

missing_required_files = required_file_check_df[
    ~required_file_check_df["exists"]
]["file"].tolist()

if missing_required_files:
    print("Missing required input files:")
    print(", ".join(missing_required_files))
else:
    print("All required input files are present.")

# Check optional external files separately.
optional_file_check_rows = []

for file_name, file_path in optional_external_files.items():
    optional_file_check_rows.append({
        "file": file_name,
        "path": str(file_path),
        "required": False,
        "exists": file_path.exists()
    })

optional_file_check_df = pd.DataFrame(optional_file_check_rows)

display(optional_file_check_df)

missing_optional_files = optional_file_check_df[
    ~optional_file_check_df["exists"]
]["file"].tolist()

if missing_optional_files:
    print(
        "\nOptional file not found yet: "
        + ", ".join(missing_optional_files)
    )
    print(
        "This is expected until you are ready to run "
        "03_match_opensanctions.ipynb."
    )
else:
    print("All optional external files are present.")

## 4. Check expected notebooks

This section checks whether the four main analysis notebooks are present.

The notebooks should be run in the following order:

1. `01_data_collection_scraper.ipynb`
2. `02_clean_build_benchmark.ipynb`
3. `03_match_opensanctions.ipynb`
4. `04_validate_benchmark_completeness.ipynb`

In [ ]:
# -------------------------------------------------------------------
# Notebook structure check
# -------------------------------------------------------------------

expected_notebooks = [
    "01_data_collection_scraper.ipynb",
    "02_clean_build_benchmark.ipynb",
    "03_match_opensanctions.ipynb",
    "04_validate_benchmark_completeness.ipynb"
]

notebook_check_rows = []

for notebook_name in expected_notebooks:
    notebook_path = NOTEBOOKS_DIR / notebook_name

    notebook_check_rows.append({
        "notebook": notebook_name,
        "path": str(notebook_path),
        "exists": notebook_path.exists()
    })

notebook_check_df = pd.DataFrame(notebook_check_rows)

display(notebook_check_df)

if notebook_check_df["exists"].all():
    print("All final analysis notebooks are present.")
else:
    print("One or more final analysis notebooks are missing.")

## 5. Setup conclusion

The project is ready for data collection when:

- the local `.venv` environment is active;
- all required packages are installed;
- all required input files are present;
- the four final notebooks are available.

The OpenSanctions file may be added later before running notebooks 03 and 04.

In [ ]:
# -------------------------------------------------------------------
# Final setup status
# -------------------------------------------------------------------

all_required_files_exist = len(missing_required_files) == 0
all_packages_installed = len(missing_packages) == 0
all_notebooks_present = notebook_check_df["exists"].all()

setup_ready = (
    using_local_venv
    and all_required_files_exist
    and all_packages_installed
    and all_notebooks_present
)

print("Setup ready:", setup_ready)

if setup_ready:
    print(
        "\nThe clean project structure is ready. "
        "Next notebook: 01_data_collection_scraper.ipynb"
    )
else:
    print(
        "\nResolve the warnings above before continuing to the data "
        "collection notebook."
    )